<a href="https://colab.research.google.com/github/AnnaShtyn/Final-Project/blob/main/%D0%A1%D1%82%D0%B2%D0%BE%D1%80%D0%B5%D0%BD%D0%BD%D1%8F_%D1%82%D0%B0%D0%B1%D0%BB%D0%B8%D1%86%D1%8C_%D1%83_BigQuery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install pandas numpy --quiet
%pip install google-cloud-bigquery db-dtypes --quiet

In [ ]:
 import pandas as pd

# зчитуємо датасет у датафрейм
df = pd.read_csv('hotel_bookings_cleaned.csv')
df.head(3)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,lead_time_category,stay_duration_category,price_tier,customer_segment,booking_type,season,is_peak_season,revenue_per_guest,average_night_value,clv_proxy
0,Resort Hotel,0,342.0,2015,July,27,1,0,0,2,...,Very Long Term (180d+),NaN,Budget,Non-Family,Standard Booking,Summer,1,0.0,0.0,0.0
1,Resort Hotel,0,444.0,2015,July,27,1,0,0,2,...,Very Long Term (180d+),NaN,Budget,Non-Family,Standard Booking,Summer,1,0.0,0.0,0.0
2,Resort Hotel,0,7.0,2015,July,27,1,0,1,1,...,Last Minute (0-7d),1 Night,Economy,Non-Family,Quick Trip,Summer,1,75.0,75.0,75.0


In [ ]:
# переглянемо колонки: лише в 2ох є відсутні значення, але оскільки вони не є основними, і нам не знадобляться, не будемо їх опрацьовувати
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119209 entries, 0 to 119208
Data columns (total 49 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119209 non-null  object 
 1   is_canceled                     119209 non-null  int64  
 2   lead_time                       119209 non-null  float64
 3   arrival_date_year               119209 non-null  int64  
 4   arrival_date_month              119209 non-null  object 
 5   arrival_date_week_number        119209 non-null  int64  
 6   arrival_date_day_of_month       119209 non-null  int64  
 7   stays_in_weekend_nights         119209 non-null  int64  
 8   stays_in_week_nights            119209 non-null  int64  
 9   adults                          119209 non-null  int64  
 10  children                        119209 non-null  float64
 11  babies                          119209 non-null  int64  
 12  country         

In [ ]:
# створюємо нову колонку 'booking_id' як primary key та структуруємо датасет з 50 колонок у 5 таблиць
df['booking_id'] = range(1, (len(df) + 1))

booking_details = df[[
    'booking_id',
    'hotel',
    'is_canceled',
    'lead_time',
    'lead_time_category',
    'arrival_date_year',
    'arrival_date_month',
    'arrival_date_week_number',
    'arrival_date_day_of_month',
    'reservation_status',
    'reservation_status_date',
    'booking_changes',
    'days_in_waiting_list',
    'booking_type'
]]

guests = df[[
    'booking_id',
    'adults',
    'children',
    'babies',
    'total_guests',
    'country',
    'country_name',
    'customer_type',
    'customer_segment',
    'is_repeated_guest',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'has_children'
]]

stay_details = df[[
    'booking_id',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'total_nights',
    'stay_duration_category',
    'assigned_room_type',
    'reserved_room_type',
    'meal_type',
    'is_weekend_stay',
    'required_car_parking_spaces',
    'total_of_special_requests',
    'has_special_requests'
]]

revenue = df[[
    'booking_id',
    'average_daily_rate',
    'total_revenue',
    'revenue_per_guest',
    'average_night_value',
    'deposit_type',
    'price_tier',
    'clv_proxy'
]]

marketing = df[[
    'booking_id',
    'market_segment',
    'distribution_channel',
    'agent',
    'is_direct_booking',
    'season',
    'is_peak_season'
]]

# зберігаємо новостворені таблиці в csv-файли
booking_details.to_csv("booking_details.csv", index=False)
guests.to_csv("guests.csv", index=False)
stay_details.to_csv("stay_details.csv", index=False)
revenue.to_csv("revenue.csv", index=False)
marketing.to_csv("marketing.csv", index=False)

In [ ]:
# підключаємось з Google Colaboratory до BigQuery
!gcloud auth application-default login

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=JHjSwnzXIcVv4UFF2fzuRZmEqK337O&prompt=consent&token_usage=remote&access_type=offline&code_challenge=S46wB-lQo20GCBomE6-AsLP-MsJwYSOz8wtDPvhuV7M&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AeoWuM9g1B7dYdrb5cxQAhckCRDRJiVl5M0zvDLIcDMCYnU39JeWKs1fYsc9oGrCWdxPvw

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Ca

In [ ]:
from google.cloud import bigquery

# завантажимо таблиці в BigQuery
PROJECT_ID = "hotel-bookings-project-498318"
DATASET    = "hotel_bookings"

client = bigquery.Client(project=PROJECT_ID)

tables = {
    "booking_details": pd.read_csv("booking_details.csv"),
    "guests":          pd.read_csv("guests.csv"),
    "stay_details":    pd.read_csv("stay_details.csv"),
    "revenue":         pd.read_csv("revenue.csv"),
    "marketing":       pd.read_csv("marketing.csv")
}

for name, table_df in tables.items():
    table_id = f"{PROJECT_ID}.{DATASET}.{name}"
    job = client.load_table_from_dataframe(
        table_df, table_id,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
    )
    job.result()
    print(f"  ✅ {name}: {client.get_table(table_id).num_rows} рядків")

    # як бачимо, всі таблиці коректно завантажились

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


  ✅ booking_details: 119209 рядків
  ✅ guests: 119209 рядків
  ✅ stay_details: 119209 рядків
  ✅ revenue: 119209 рядків
  ✅ marketing: 119209 рядків
